## 1. 生成数据与概率分布

In [1]:
import importlib
import numpy as np

import data as data_mod
import evaluator as eval_mod
import solver as solver_mod

importlib.reload(data_mod)
importlib.reload(eval_mod)
importlib.reload(solver_mod)

from data import (
    generate_data,
    compute_posterior_distribution,
    sample_from_mapping,
    compute_y_marginal_stats,
)
from evaluator import mutual_information, evaluate_transmission
from solver import (
    agglomerative_ib,
    sequential_ib,
    build_cluster_decode_table,
    nearest_centroid_predict,
    knn_predict,
    svm_fit_predict,
)

# ==================== 超参数配置 ====================
N_CLUSTERS = 80       # 聚类后的簇数 K（可按需修改）
N_X = 1000             # 训练样本数
X_DIM = 10            # X 向量维度
Y_DIM = 10            # Y 向量维度
N_TEST = 500          # 测试新样本数
KNN_K = 5             # KNN 近邻数
SVM_C = 10.0          # SVM 惩罚系数 C
SVM_GAMMA = None      # RBF 核参数 gamma（None 表示 1 / X_DIM）
SEED = 1989            # 训练数据随机种子
TEST_SEED = 64       # 测试数据随机种子
# ====================================================

data = generate_data(
    n_x=N_X,
    x_dim=X_DIM,
    x_low=0,
    x_high=10,
    y_dim=Y_DIM,
    max_degree=3,
    seed=SEED,
)

X = data["X"]
P_y_given_x = data["prob_matrix"]

p_xy, p_x, p_y, p_y_given_x_ib = compute_posterior_distribution(P_y_given_x)
i_xy = mutual_information(p_xy)

print(f"X shape: {X.shape}, P(Y|X) shape: {P_y_given_x.shape}, 目标聚类簇数 K = {N_CLUSTERS}")
print(f"原始互信息 I(X; Y) = {i_xy:.4f} bits")

# 统计各维度与总计的 P(Y_i = 1)，确认正负样本分布均衡、后续检验有意义
y_stats = compute_y_marginal_stats(P_y_given_x, threshold=0.5)
print(f"\n=== 训练集边际分布统计 P(Y_i = 1) ===")
print(f"总计平均 P(Y=1) = {y_stats['p_y_total']:.4f} | P(Y|X)>0.5 总占比 = {y_stats['pos_ratio_total']:.2%} | 多数类基线 = {y_stats['majority_baseline_total']:.2%}")
for j in range(Y_DIM):
    print(
        f"  Y[{j}]: P(Y_{j}=1) = {y_stats['p_y_i'][j]:.4f} | "
        f"P>0.5占比 = {y_stats['pos_ratio_i'][j]:.2%} | "
        f"多数类基线 = {y_stats['majority_baseline_i'][j]:.2%}"
    )
imbalanced = np.where((y_stats['pos_ratio_i'] < 0.25) | (y_stats['pos_ratio_i'] > 0.75))[0]
if len(imbalanced) > 0:
    print(f"[提示] 维度 {imbalanced.tolist()} 类别较不均衡，可调整 SEED 或 logit_scale。")
else:
    print("所有 Y_i 维度分布均衡（P>0.5 占比均在 25%~75% 之间），适合后续检验。")

X shape: (1000, 10), P(Y|X) shape: (1000, 10), 目标聚类簇数 K = 80
原始互信息 I(X; Y) = 0.2117 bits

=== 训练集边际分布统计 P(Y_i = 1) ===
总计平均 P(Y=1) = 0.5032 | P(Y|X)>0.5 总占比 = 50.37% | 多数类基线 = 64.13%
  Y[0]: P(Y_0=1) = 0.3913 | P>0.5占比 = 35.50% | 多数类基线 = 64.50%
  Y[1]: P(Y_1=1) = 0.5546 | P>0.5占比 = 59.20% | 多数类基线 = 59.20%
  Y[2]: P(Y_2=1) = 0.5097 | P>0.5占比 = 50.30% | 多数类基线 = 50.30%
  Y[3]: P(Y_3=1) = 0.3118 | P>0.5占比 = 19.70% | 多数类基线 = 80.30%
  Y[4]: P(Y_4=1) = 0.6273 | P>0.5占比 = 68.70% | 多数类基线 = 68.70%
  Y[5]: P(Y_5=1) = 0.7341 | P>0.5占比 = 85.00% | 多数类基线 = 85.00%
  Y[6]: P(Y_6=1) = 0.5609 | P>0.5占比 = 59.30% | 多数类基线 = 59.30%
  Y[7]: P(Y_7=1) = 0.4769 | P>0.5占比 = 47.70% | 多数类基线 = 52.30%
  Y[8]: P(Y_8=1) = 0.4032 | P>0.5占比 = 34.00% | 多数类基线 = 66.00%
  Y[9]: P(Y_9=1) = 0.4624 | P>0.5占比 = 44.30% | 多数类基线 = 55.70%
[提示] 维度 [3, 5] 类别较不均衡，可调整 SEED 或 logit_scale。


## 2. 凝聚式信息瓶颈（aIB）

In [2]:
aib_result = agglomerative_ib(
    p_x=p_x,
    p_y_given_x=p_y_given_x_ib,
    n_clusters=N_CLUSTERS,
    record_full_hierarchy=True,
)
aib_labels = aib_result["labels"]
aib_i_ty = aib_result["i_ty"]

print(f"aIB 聚类完成: 簇数 = {len(np.unique(aib_labels))}, 样本标签 shape = {aib_labels.shape}")
print(f"aIB 各簇样本数: {np.bincount(aib_labels, minlength=N_CLUSTERS)}")
print(f"aIB 互信息 I(T; Y) = {aib_i_ty:.4f} bits (保留比例 {aib_i_ty / i_xy:.2%})")

aIB 聚类完成: 簇数 = 80, 样本标签 shape = (1000,)
aIB 各簇样本数: [11 16 19 11 15 11  8 18  7 14 20 24 11 27 16  7 16 12 14  7  8 10 15 18
  8 11 14 19 15 13  9 12 26 10 28 12 13 12 17 19 10 11 12  8 29  9 13 21
 16 15  8 23 15 14 12  8 12  8 11  5  6  8  5 18  8  8  8  7  5 13  9  7
  6  6 11  7 10  9 10  6]
aIB 互信息 I(T; Y) = 0.1387 bits (保留比例 65.51%)


## 3. 序列信息瓶颈（sIB）

In [3]:
sib_result = sequential_ib(
    p_x=p_x,
    p_y_given_x=p_y_given_x_ib,
    n_clusters=N_CLUSTERS,
    max_iter=100,
    n_init=10,
    seed=SEED,
)
sib_labels = sib_result["labels"]
sib_i_ty = sib_result["i_ty"]

# 汇总存放两种算法的聚类结果与后验分布 P(Y|T)，供后续各映射模块使用
clustering_results = {
    "aIB": aib_labels,
    "sIB": sib_labels,
}
cluster_tables = {}
for name, labels in clustering_results.items():
    centroids, y_decode, p_y_given_t = build_cluster_decode_table(
        X, P_y_given_x, labels, n_clusters=N_CLUSTERS
    )
    cluster_tables[name] = {
        "T_centroids": centroids,
        "y_decode_table": y_decode,
        "p_y_given_t": p_y_given_t,
    }

print(f"sIB 聚类完成: 簇数 = {len(np.unique(sib_labels))}, 样本标签 shape = {sib_labels.shape}, 收敛轮数 = {len(sib_result['mi_history']) - 1}")
print(f"sIB 各簇样本数: {np.bincount(sib_labels, minlength=N_CLUSTERS)}")
print(f"sIB 互信息 I(T; Y) = {sib_i_ty:.4f} bits (保留比例 {sib_i_ty / i_xy:.2%})")

sIB 聚类完成: 簇数 = 80, 样本标签 shape = (1000,), 收敛轮数 = 18
sIB 各簇样本数: [ 8 17 14 17 14 10 13 18 11 12 17 14 11 13 14 11  9 13 14 16 13 11 11 11
  8  8 23 10 13 17  5 13  9 12 14 15 11 10 11 20 11  8 11 11 19 15  8 21
 15 11 11  6 16 13 12 14 15 24 16 11  8 14 22  9 11 12  6  7  7  5  9 17
 13 13 15 13  9  9 10 12]
sIB 互信息 I(T; Y) = 0.1436 bits (保留比例 67.81%)


## 4. 最近质心距离映射

In [4]:
# 生成新测试样本（沿用同一多项式映射计算真实后验 P(Y|X_test)）
test_data = sample_from_mapping(data, n_samples=N_TEST, seed=TEST_SEED)
X_test = test_data["X"]
P_test = test_data["prob_matrix"]
test_y_stats = compute_y_marginal_stats(P_test, threshold=0.5)
print(
    f"测试集边际统计: 总计平均 P(Y=1) = {test_y_stats['p_y_total']:.4f} | "
    f"P(Y|X)>0.5 总占比 = {test_y_stats['pos_ratio_total']:.2%} | "
    f"多数类基线 = {test_y_stats['majority_baseline_total']:.2%}\n"
)

nearest_centroid_results = {}
for method_name in clustering_results:
    T_centroids = cluster_tables[method_name]["T_centroids"]
    p_y_given_t = cluster_tables[method_name]["p_y_given_t"]

    pred_idx = nearest_centroid_predict(X_test, T_centroids)
    T_test = T_centroids[pred_idx]
    P_pred = p_y_given_t[pred_idx]

    metrics = evaluate_transmission(P_pred, P_test)
    nearest_centroid_results[method_name] = {"T_test": T_test, "P_pred": P_pred, **metrics}

    print(f"=== {method_name} 最近质心映射测试结果 (N_TEST={N_TEST}, K={N_CLUSTERS}) ===")
    print(f"总体还原率: {metrics['overall_acc']:.2%} ({metrics['correct_count']}/{metrics['total_count']})")
    print()

测试集边际统计: 总计平均 P(Y=1) = 0.4986 | P(Y|X)>0.5 总占比 = 50.06% | 多数类基线 = 65.10%

=== aIB 最近质心映射测试结果 (N_TEST=500, K=80) ===
总体还原率: 75.18% (3759/5000)

=== sIB 最近质心映射测试结果 (N_TEST=500, K=80) ===
总体还原率: 76.06% (3803/5000)



## 5. KNN映射

In [5]:
knn_results = {}
for method_name, labels in clustering_results.items():
    T_centroids = cluster_tables[method_name]["T_centroids"]
    p_y_given_t = cluster_tables[method_name]["p_y_given_t"]

    train_pred_idx, test_pred_idx = knn_predict(
        X, labels, X_test, n_clusters=N_CLUSTERS, n_neighbors=KNN_K, weighted=True
    )
    train_cls_acc = float((train_pred_idx == labels).mean())
    T_test = T_centroids[test_pred_idx]
    P_pred = p_y_given_t[test_pred_idx]

    metrics = evaluate_transmission(P_pred, P_test)
    knn_results[method_name] = {
        "train_cls_acc": train_cls_acc,
        "T_test": T_test,
        "P_pred": P_pred,
        **metrics,
    }

    print(f"=== {method_name} KNN映射测试结果 (k={KNN_K}, N_TEST={N_TEST}, K={N_CLUSTERS}) ===")
    print(f"训练集簇拟合一致率: {train_cls_acc:.2%}")
    print(f"总体还原率: {metrics['overall_acc']:.2%} ({metrics['correct_count']}/{metrics['total_count']})")
    print()

=== aIB KNN映射测试结果 (k=5, N_TEST=500, K=80) ===
训练集簇拟合一致率: 100.00%
总体还原率: 75.34% (3767/5000)

=== sIB KNN映射测试结果 (k=5, N_TEST=500, K=80) ===
训练集簇拟合一致率: 100.00%
总体还原率: 75.58% (3779/5000)



## 6. SVM映射

In [6]:
svm_results = {}
for method_name, labels in clustering_results.items():
    T_centroids = cluster_tables[method_name]["T_centroids"]
    p_y_given_t = cluster_tables[method_name]["p_y_given_t"]

    train_pred_idx, test_pred_idx = svm_fit_predict(
        X, labels, X_test, n_clusters=N_CLUSTERS, C=SVM_C, gamma=SVM_GAMMA
    )
    train_cls_acc = float((train_pred_idx == labels).mean())
    T_test = T_centroids[test_pred_idx]
    P_pred = p_y_given_t[test_pred_idx]

    metrics = evaluate_transmission(P_pred, P_test)
    svm_results[method_name] = {
        "train_cls_acc": train_cls_acc,
        "T_test": T_test,
        "P_pred": P_pred,
        **metrics,
    }

    print(f"=== {method_name} SVM映射测试结果 (C={SVM_C}, N_TEST={N_TEST}, K={N_CLUSTERS}) ===")
    print(f"训练集簇拟合一致率: {train_cls_acc:.2%}")
    print(f"总体还原率: {metrics['overall_acc']:.2%} ({metrics['correct_count']}/{metrics['total_count']})")
    print()

# ==================== 汇总对比 ====================
print("=== 各映射方式总体还原率汇总 (Y|T 与 Y|X 在 0.5 同侧即正确) ===")
header = f"{'组合':<18} | {'总体还原率':>10} | {'正确数/总数':>12}"
print(header)
print("-" * len(header))
summary_rows = [
    ("aIB + 最近质心", nearest_centroid_results["aIB"]),
    ("sIB + 最近质心", nearest_centroid_results["sIB"]),
    ("aIB + KNN", knn_results["aIB"]),
    ("sIB + KNN", knn_results["sIB"]),
    ("aIB + SVM", svm_results["aIB"]),
    ("sIB + SVM", svm_results["sIB"]),
]
for label_name, m in summary_rows:
    ratio_str = f"{m['correct_count']}/{m['total_count']}"
    print(f"{label_name:<18} | {m['overall_acc']:>9.2%} | {ratio_str:>12}")
print("-" * len(header))
print(f"{'测试集多数类基线':<16} | {test_y_stats['majority_baseline_total']:>9.2%} | {'(各维度盲猜众数)':>12}")

=== aIB SVM映射测试结果 (C=10.0, N_TEST=500, K=80) ===
训练集簇拟合一致率: 100.00%
总体还原率: 77.22% (3861/5000)

=== sIB SVM映射测试结果 (C=10.0, N_TEST=500, K=80) ===
训练集簇拟合一致率: 100.00%
总体还原率: 77.44% (3872/5000)

=== 各映射方式总体还原率汇总 (Y|T 与 Y|X 在 0.5 同侧即正确) ===
组合                 |      总体还原率 |       正确数/总数
----------------------------------------------
aIB + 最近质心         |    75.18% |    3759/5000
sIB + 最近质心         |    76.06% |    3803/5000
aIB + KNN          |    75.34% |    3767/5000
sIB + KNN          |    75.58% |    3779/5000
aIB + SVM          |    77.22% |    3861/5000
sIB + SVM          |    77.44% |    3872/5000
----------------------------------------------
测试集多数类基线         |    65.10% |    (各维度盲猜众数)
